In [1]:
from google.cloud import storage
import pandas as pd
import joblib
import json
from sklearn.metrics import accuracy_score
import os

In [2]:
INPUT_BUCKET = "iris-training-data-bucket"
OUTPUT_BUCKET = "mlops-course-week1-unique"

storage_client = storage.Client()

artifact_bucket = storage_client.bucket(OUTPUT_BUCKET)

In [3]:
prefixes = sorted(
    set(
        blob.name.split("/")[0]
        for blob in artifact_bucket.list_blobs()
    )
)

latest_run = prefixes[-1]

In [4]:
tmp_dir = "/tmp/inference"
os.makedirs(tmp_dir, exist_ok=True)

model_file = f"{tmp_dir}/model.joblib"

artifact_bucket.blob(
    f"{latest_run}/model.joblib"
).download_to_filename(model_file)

model = joblib.load(model_file)

In [5]:
data_bucket = storage_client.bucket(INPUT_BUCKET)

eval_file = f"iris_eval.csv"

data_bucket.blob(
    "iris_eval.csv"
).download_to_filename(eval_file)

eval_df = pd.read_csv(eval_file)

In [6]:
X_eval = eval_df.drop("species", axis=1)
y_eval = eval_df["species"]

preds = model.predict(X_eval)

accuracy = accuracy_score(y_eval, preds)

results = {
    "eval_accuracy": float(accuracy)
}
print(results)

{'eval_accuracy': 0.9833333333333333}
